In [15]:
import importlib
import icloud_auth
import course_reader
import calendar_reader
from datetime import datetime, timedelta, time as dtime
import task_reader
import agent
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyicloud.services.calendar import EventObject

importlib.reload(icloud_auth)
importlib.reload(course_reader)
importlib.reload(calendar_reader)
importlib.reload(task_reader)
importlib.reload(agent)

<module 'agent' from 'c:\\Users\\nicol\\projects\\A1\\agent.py'>

In [ ]:
from icloud_auth import start_icloud_login

icloud_state = start_icloud_login()


Text(value='', description='Apple ID:', layout=Layout(width='450px'), placeholder='you@icloud.com')

Password(description='Password:', layout=Layout(width='450px'))

Button(button_style='primary', description='Login', style=ButtonStyle())

Text(value='', description='2FA Code:', layout=Layout(display='none', width='250px'), placeholder='6-digit cod…

Button(button_style='warning', description='Submit 2FA', layout=Layout(display='none'), style=ButtonStyle())

Output()

In [ ]:

semester_state = course_reader.select_semester()

In [4]:
if "course_grades" not in semester_state or semester_state["course_grades"] is None:
    raise RuntimeError("Select a semester and click Select first.")

In [5]:
start_picker = widgets.TimePicker(
    description='Sleep Start Time:',
    style={"description_width": "150px"}
)
end_picker = widgets.TimePicker(
    description='Sleep End Time:',
    style={"description_width": "150px"}
)
day_dropdown = widgets.SelectMultiple(
    options=['Monday', 'Tuesday', 'Wednesday','Thurday', 'Friday', 'Saturday', 'Sunday'],
    description='Day off?',
    disabled=False,
    rows=7
)

finalize_btn = widgets.Button(
description="Select",
button_style="success"
)

display(start_picker, end_picker, day_dropdown, finalize_btn)

TimePicker(value=None, description='Sleep Start Time:', step=60.0, style=DescriptionStyle(description_width='1…

TimePicker(value=None, description='Sleep End Time:', step=60.0, style=DescriptionStyle(description_width='150…

SelectMultiple(description='Day off?', options=('Monday', 'Tuesday', 'Wednesday', 'Thurday', 'Friday', 'Saturd…

Button(button_style='success', description='Select', style=ButtonStyle())

In [6]:
api = icloud_state["api"]
events = calendar_reader.grab_events(api)
calendar_busy = calendar_reader.events_to_busy_intervals(events)

sleep_start = dtime(22, 0)
sleep_end = dtime(6,0)
if start_picker.value is  not None or end_picker.value is not None:
    sleep_start=start_picker.value
    sleep_end=end_picker.value

sleep_busy = calendar_reader.build_sleep_intervals(
    days=7,
    sleep_start = sleep_start,
    sleep_end = sleep_end,
    days_off = day_dropdown.value
)


In [7]:
# 1. Use the grades already computed during semester selection
course_grades = semester_state["course_grades"]

# 2. Normalize once
normalized_grades = course_reader.normalize_course_grades(course_grades, zero_means_missing=False)

# 3. Compute risk from normalized grades
course_risk_weights = course_reader.course_risk_weights(normalized_grades)

# 4. Read open tasks correctly (semester name + base dir)
tasks_by_course = task_reader.read_all_tasks(
    semester=semester_state["semester"],
    base_dir=semester_state["base_dir"]
    )

# 5. Build final agent input
agent_input = {
    "semester": semester_state["semester"],
    "course_grades": normalized_grades,
    "course_risk_weights": course_risk_weights,
    "busy_intervals": calendar_reader.merge_intervals(calendar_busy + sleep_busy),
    "tasks_by_course": tasks_by_course,
}
agent.validate_agent_input(agent_input)

agent_input schema is valid


In [8]:
start_picker = widgets.DatetimePicker(
    description="Start Study Block:"
)

end_picker = widgets.DatetimePicker(
    description="End Study Block:"
)

finalize_btn = widgets.Button(
description="Select",
button_style="success"
)

display(start_picker, end_picker, finalize_btn)


DatetimePicker(value=None, description='Start Study Block:')

DatetimePicker(value=None, description='End Study Block:')

Button(button_style='success', description='Select', style=ButtonStyle())

In [16]:
for i, (b0, b1) in enumerate(agent_input["busy_intervals"]):
    if b0.tzinfo is None or b1.tzinfo is None:
        print("Naive busy interval at index", i, b0, b1)
        break


In [17]:
tz = agent_input["busy_intervals"][0][0].tzinfo
time_start = datetime.now(tz).date
time_end = datetime.now(tz) + timedelta(days=7)
if start_picker.value is  not None or end_picker.value is not None:
    time_start=start_picker.value
    time_end=end_picker.value

tz_ref = agent_input["busy_intervals"][0][0].tzinfo

if time_start.tzinfo is None:
    time_start = time_start.replace(tzinfo=tz_ref)
else:
    time_start = time_start.astimezone(tz_ref)

if time_end.tzinfo is None:
    time_end = time_end.replace(tzinfo=tz_ref)
else:
    time_end = time_end.astimezone(tz_ref)


In [18]:
out = widgets.Output()

study_plan = agent.recommend_study_plan(agent_input, time_start, time_end)

schedules = study_plan["schedule"]
calendar_service = icloud_state["api"].calendar
calendars = calendar_service.get_calendars(as_objs=True)
calendar_guid = calendars[1].guid

In [22]:
display(out)

def create_calendar_event(title, start, end):
    event = EventObject(
        pguid=calendar_guid,
        title=title,
        start_date=start,
        end_date=end,
        all_day=False
    )
    calendar_service.add_event(event)
    
    
approve_btn = widgets.Button(description="Approve", button_style='success')
skip_btn = widgets.Button(description = "Skip", button_style = 'warning')
edit_btn = widgets.Button(description = "Edit")
stop_btn = widgets.Button(description = "Stop", button_style = 'danger')

button_row = widgets.HBox([approve_btn, skip_btn, edit_btn, stop_btn])

current_index = 0
approved = []
skipped = []
stopped = False
def render_current():
    with out:
        clear_output(wait=True)

        if current_index >= len(schedules):
            print("Done — no more recommendations.")
            return

        item = schedules[current_index]
        start = item["start"]
        end = item["end"]
        due = item.get("due")

        print(f"[{current_index+1}/{len(schedules)}] {item['course']} : {item['title']}")
        if due is None:
            print("No assigned due date.")
        else:
            print("When task is due:", due.strftime("%a %b %d, %Y at %I:%M %p"))
        print("When to study:", start.strftime("%a %b %d, %Y"))
        print(f"{start.strftime('%I:%M %p')} - {end.strftime('%I: %M %p')}")
        print(f"Blocked off minutes: {item.get('block_minutes', '?')} | Tasks total minutes: {item.get('total_minutes','?')}")
        print("-" * 50)

        display(button_row)

render_current()

def on_approve_clicked(b):
    global current_index

    item = schedules[current_index]

    # ---- INSERT INTO CALENDAR HERE ----
    create_calendar_event(
        title=f"{item['course']} - {item['title']}",
        start=item["start"],
        end=item["end"],
    )

    approved.append(item)
    current_index += 1
    render_current()

def on_skip_clicked(b):
    global current_index

    skipped.append(schedules[current_index])
    current_index += 1
    render_current()

def on_stop_clicked(b):
    with out:
        clear_output()
        print("Review stopped.")
        print(f"Approved so far: {len(approved)}")
        print(f"Skipped so far: {len(skipped)}")

new_start_picker = widgets.DatetimePicker(description="New Start")
duration_box = widgets.IntText(description="Minutes")
save_edit_btn = widgets.Button(description="Save Edit", button_style="success")
cancel_edit_btn = widgets.Button(description="Cancel")

def on_edit_clicked(b):
    with out:
        clear_output(wait=True)

        item = schedules[current_index]

        new_start_picker.value = item["start"]
        duration_box.value = item["block_minutes"]

        display(new_start_picker)
        display(duration_box)
        display(widgets.HBox([save_edit_btn, cancel_edit_btn]))

def on_save_edit_clicked(b):
    global current_index

    item = schedules[current_index]

    new_start = new_start_picker.value
    new_duration = duration_box.value
    new_end = new_start + timedelta(minutes=new_duration)

    item["start"] = new_start
    item["end"] = new_end
    item["block_minutes"] = new_duration

    approved.append(item)

    create_calendar_event(
        title=f"Study: {item['course']} - {item['title']}",
        start=item["start"],
        end=item["end"]
    )

    current_index += 1
    render_current()
def on_cancel_edit_clicked(b):
    render_current()

approve_btn.on_click(on_approve_clicked)
skip_btn.on_click(on_skip_clicked)
edit_btn.on_click(on_edit_clicked)
stop_btn.on_click(on_stop_clicked)

save_edit_btn.on_click(on_save_edit_clicked)
cancel_edit_btn.on_click(on_cancel_edit_clicked)

Output()